In [ ]:
import json
import re
import pandas as pd

from unstract.llmwhisperer import LLMWhispererClientV2
from unstract.llmwhisperer.client_v2 import LLMWhispererClientException

In [ ]:
import os
from pathlib import Path
client = LLMWhispererClientV2(api_key=os.environ["LLMWHISPERER_API_KEY"])  # never hard-code a real key here

try:
    # Procesa el archivo PDF y espera a que termine (wait_for_completion=True)
    result = client.whisper(
        file_path = str(Path('..') / 'dados' / 'djos' / 'DJO 4.pdf')#"/content/drive/MyDrive/Ferramenta IA - Certificados de Origem/djos/DJO 2 (2).pdf"
,
        wait_for_completion=True,
        wait_timeout=200
    )

    # Extrae el texto obtenido del resultado
    extracted_text = result["extraction"]["result_text"]
    print("Texto extraído con éxito:")
    print(extracted_text)

except LLMWhispererClientException as e:
    print(f"Error en la API: {e.message}, Código de estado: {e.status_code}")

In [ ]:
extracted_text

In [77]:
# =============================================================================
# Parser SEQUENCIAL do texto OCR (LLM Whisperer, layout_preserving) da DJO
# =============================================================================
#
# *** NOTA: esta célula estava tendo suas edições revertidas para uma versão
# antiga (baseada em extract_simple_field) sempre que o notebook era salvo a
# partir de um buffer desatualizado no editor. Se "Razão social do produtor"
# voltar a aparecer como uma STRING simples em vez de um dicionário com
# {"Razão social", "CNPJ/CPF", "Inscrição Estadual"}, é sinal de que isso
# aconteceu de novo — comparem com o que está descrito abaixo antes de mexer.
#
# As chaves do JSON são âncoras/delimitadores. Caminhamos pelo texto do OCR
# UMA ÚNICA VEZ, sempre para frente (nunca com buscas independentes por todo
# o texto para cada campo). Para a chave atual, tudo que vem depois dela e
# ANTES da PRÓXIMA chave da lista é o "bloco bruto" daquele campo — não
# importa quantas linhas ele ocupe. Isso resolve, por construção:
#   - só pegar a 1ª linha depois do rótulo ("Razão social do produtor" e
#     "Razão social do exportador" têm até 3 linhas: nome, CNPJ/CPF,
#     Inscrição Estadual — todas capturadas por handler_empresa, que só
#     preenche os campos que existirem de fato, nunca inventa os que faltam);
#   - confundir o rótulo "Domicílio legal e parque industrial:" repetido
#     (produtor E exportador usam o mesmo texto) — como cada ocorrência do
#     rótulo é consumida em sequência, a 1ª aparição vira sempre o domicílio
#     do produtor e a 2ª o do exportador, sem precisar de nenhum caso especial.
#
# 0) PRÉ-PROCESSAMENTO (preprocess_ocr_text)
#    - "Código da aprovação DJO" se repete como cabeçalho em toda página;
#      varremos TODAS as ocorrências e usamos a primeira que tiver um valor
#      de fato (algumas DJOs deixam em branco na página 1 e só preenchem na
#      página 2; outras já trazem o código logo na 1ª ocorrência, na MESMA
#      linha do rótulo — os dois casos são tratados). As demais ocorrências
#      são só repetição e nunca viram um 2º valor; são removidas do texto
#      (inclusive quando o valor está colado na mesma linha do rótulo, para
#      não vazar para dentro do bloco do campo seguinte).
#    - remove o título repetido "DECLARAÇÃO JURAMENTADA DE ORIGEM";
#    - remove o parágrafo padrão de declaração legal + assinaturas
#      ("DECLARO PARA OS DEVIDOS FINS...");
#    - remove os rodapés "Página X de Y";
#    - troca o separador de página "<<<" por uma quebra de linha comum (não
#      corta o texto: DJOs de 3 páginas podem ter tabelas continuando).
#
# 1) MOTOR SEQUENCIAL (sequential_extract) + LISTA DE CHAVES (build_keys)
#    Cada campo é uma "chave" com um regex de rótulo e um "handler" que
#    transforma o bloco bruto no valor final. `merge=True` é usado só para
#    "Código NCM - NALADI: ... Denominação comercial", que produz 3 campos
#    de uma vez.
#
# 2) TABELAS DE MATERIAIS (handler_tabela / _parse_item_row)
#    Descrição / País origem continuam separados por espaçamento (blocos de
#    2+ espaços). Valor (US$) / % S/VALOR FOB / FORNECEDOR-FABRICANTE são
#    separados por CONTEÚDO, não por espaçamento — regra pedida
#    explicitamente: qualquer valor que bata com o padrão de porcentagem
#    (0%, 5%, 12.5%, 25,50%, 100%...) é o "% S/VALOR FOB"; tudo que sobra
#    depois dele é "FORNECEDOR/FABRICANTE", mesmo colado sem espaço nenhum
#    ou grudado num texto padrão como "0% DISPONIBLE À SOLITUD DE LAS
#    AUTORIDADES COMPETENTES" (esse é o placeholder do formulário quando o
#    fornecedor não é divulgado — vira só "0%" + "DISPONIBLE À SOLITUD DE
#    LAS AUTORIDADES COMPETENTES", nunca as duas coisas juntas em "% S/VALOR
#    FOB"). Concretamente: localizamos o ÚLTIMO padrão "número%" da linha —
#    inconfundível por causa do símbolo "%" — e tudo depois dele é
#    Fornecedor/Fabricante. Do que sobra antes do "%", pegamos o último
#    número (ou "[Não Informado]") como Valor (US$); o resto é Descrição +
#    País origem. Linhas de continuação (quando uma célula quebra em 2
#    linhas) são associadas à coluna cuja âncora de posição estiver mais
#    próxima.
#
#    *** "Somatório" tolerante a erro de OCR na vogal acentuada: em alguns
#    documentos a OCR grafa "Somatório" como "Somatário" (ó -> á lido
#    errado). Antes disso quebrar o parser: a linha do total não era
#    reconhecida como "pare aqui", e acabava sendo tratada como uma linha de
#    continuação do último item, grudando "Somatário: 6%" dentro de
#    "% s/Valor FOB" (e "Somatório" do resultado ficava vazio, porque a
#    extração do total também exigia a grafia exata "Somatório"/"Somatorio").
#    Agora o regex usa "Somat.rio" (qualquer caractere no lugar da vogal
#    acentuada) nos 3 lugares que precisam reconhecer essa palavra. Além
#    disso, como blindagem: ao final da tabela, qualquer texto que tenha
#    vazado para depois do valor percentual em "% s/Valor FOB" é descartado
#    — essa coluna só deve conter um número e o sinal "%".
#
#    *** Descrição/País origem grudados por um ÚNICO espaço (sem o gap de
#    2+ espaços que normalmente separa as colunas): acontecia, por exemplo,
#    em "...iodo. lodo. Outros. CHILE" — a OCR não deixou espaço duplo antes
#    de "CHILE", então a coluna inteira virava um token só e "CHILE" ficava
#    dentro de "Descrição", com "País origem" vazio. _parse_item_row agora
#    tenta, só nesse caso (token único), separar um nome de país conhecido
#    (KNOWN_COUNTRIES, sempre em maiúsculas nesses documentos) do final do
#    texto grudado. Se o país não estiver na lista, o comportamento cai de
#    volta exatamente no que já existia antes desta correção (tudo em
#    Descrição, País origem vazio) — nunca fica pior do que estava.
#
# 3) DEBUG (ver célula seguinte)
#    `parsear_djo` devolve (json_final, debug_blocks) — debug_blocks é a
#    lista, na ordem do documento, de cada bloco bruto extraído antes de
#    virar valor final.

import re

_LABEL_PATTERNS = [
    r"Código da aprovação DJO:",
    r"Número da DJO:",
    r"Data de apresentação:",
    r"Razão social do produtor:",
    r"CNPJ/CPF:",
    r"Inscrição Estadual:",
    r"Domicílio legal e parque industrial:",
    r"Endereço:",
    r"Telefone:",
    r"E-mail:",
    r"Razão social do exportador:",
    r"Código NCM\s*-\s*NALADI:",
    r"Denominação comercial do produto a exportar",
    r"Valor FOB \(USD\):",
    r"Unidade de medida:",
    r"Descrição do processo produtivo:",
    r"Materiais utilizados:",
    r"Materiais originários do Estado Parte produtor:",
    r"Materiais originários de outros Estados Partes:",
    r"Materiais não originários:",
    r"Materiais de terceiros países que tenham cumprido com a PTC:",
    r"Porcentagem Total de Mat[eé]rias Primas",
    r"Valor agregado no processo Industrial",
    r"Preço FOB:",
    r"Observações:",
    r"Somat.rio",
    r"Página\s*\d+\s*de\s*\d+",
    r"DECLARAÇÃO JURAMENTADA DE ORIGEM",
    r"NCM/SH",
    r"DECLARO PARA OS DEVIDOS FINS",
]
_LABEL_RE = re.compile(r"^\s*(?:" + "|".join(_LABEL_PATTERNS) + r")", re.IGNORECASE)


def _is_label_line(line):
    """True se a linha (já sem espaços nas pontas) é, ela mesma, um dos
    rótulos conhecidos do formulário DJO — usado para saber quando um bloco
    "vazio" termina, em vez de engolir o próximo rótulo como valor."""
    return bool(_LABEL_RE.match(line))


# ---------------------------------------------------------------------------
# 0) PRÉ-PROCESSAMENTO
# ---------------------------------------------------------------------------
def preprocess_ocr_text(extracted_text):
    """Retorna (codigo_aprovacao_djo, texto_limpo_para_extracao_sequencial)."""
    texto = extracted_text

    # --- "Código da aprovação DJO": pega a 1ª ocorrência que tiver valor
    # (o valor pode vir na mesma linha do rótulo OU na linha seguinte)
    codigo_aprovacao = ""
    for m in re.finditer(r"Código da aprovação DJO:", texto):
        for linha in texto[m.end():].split("\n"):
            s = linha.strip()
            if not s:
                continue
            if s == "[Não Informado]" or _is_label_line(s):
                break  # esta ocorrência não tem valor -> tenta a próxima
            codigo_aprovacao = s
            break
        if codigo_aprovacao:
            break

    # remove TODAS as ocorrências do rótulo — inclusive quando o valor vem
    # colado na mesma linha (`[^\n]*` consome esse resto da linha) — para
    # que nenhuma seja reinterpretada como um novo valor, nem vaze para
    # dentro do bloco do campo seguinte.
    texto = re.sub(
        r"Código da aprovação DJO:[^\n]*\n(?:\s*\n)?(?:[ \t]*\[Não Informado\][ \t]*\n)?",
        "\n",
        texto,
    )

    # remove o título repetido em cada página
    texto = re.sub(r"DECLARAÇÃO JURAMENTADA DE ORIGEM", "\n", texto)

    # remove o parágrafo de declaração legal + assinaturas (texto padrão,
    # idêntico em todas as DJOs), até o próximo rodapé de página (ou até o
    # fim do texto, se não houver mais rodapé)
    texto = re.sub(
        r"DECLARO PARA OS DEVIDOS FINS.*?(?=Página\s*\d+\s*de\s*\d+|$)",
        "\n",
        texto,
        flags=re.DOTALL,
    )

    # remove os rodapés de página
    texto = re.sub(r"Página\s*\d+\s*de\s*\d+", "\n", texto)

    # separador de página -> só uma quebra de linha (não corta o documento:
    # DJOs de 3 páginas podem ter tabelas continuando na página seguinte)
    texto = texto.replace("<<<", "\n")

    return codigo_aprovacao, texto


# ---------------------------------------------------------------------------
# 1) MOTOR DE EXTRAÇÃO SEQUENCIAL
# ---------------------------------------------------------------------------
def sequential_extract(texto, keys):
    """
    keys: lista de dicts com:
        name    -> identificador interno do campo
        pattern -> regex que casa o rótulo no texto
        handler -> função(bloco_bruto) -> valor, ou None (chave estrutural,
                   sem valor a guardar, mas que ainda aparece no debug)
        merge   -> se True, o retorno do handler (um dict) é mesclado no
                   nível superior do resultado, em vez de aninhado sob `name`
    Retorna (resultado, blocos_debug), com blocos_debug na ordem do documento.
    """
    resultado = {}
    blocos_debug = []
    cursor = 0

    for i, key in enumerate(keys):
        m = re.search(key["pattern"], texto[cursor:], re.IGNORECASE)
        if not m:
            blocos_debug.append({"key": key["name"], "found": False, "matched_label": None, "raw_block": ""})
            continue

        value_start = cursor + m.end()
        matched_label = m.group(0)

        # delimita o bloco pela PRÓXIMA chave que realmente existir no texto
        next_start = len(texto)
        for key_seguinte in keys[i + 1:]:
            m2 = re.search(key_seguinte["pattern"], texto[value_start:], re.IGNORECASE)
            if m2:
                next_start = value_start + m2.start()
                break

        raw_block = texto[value_start:next_start]
        blocos_debug.append({
            "key": key["name"], "found": True,
            "matched_label": matched_label.strip(), "raw_block": raw_block,
        })

        handler = key.get("handler")
        if handler is not None:
            valor = handler(raw_block)
            if key.get("merge"):
                resultado.update(valor)
            else:
                resultado[key["name"]] = valor

        cursor = value_start

    return resultado, blocos_debug


# ---------------------------------------------------------------------------
# 2) HANDLERS DE CAMPO
# ---------------------------------------------------------------------------
def _texto_limpo(bloco):
    """Junta um bloco (potencialmente multi-linha) num único texto,
    colapsando espaços/quebras de linha. Serve tanto para campos de 1 linha
    quanto para parágrafos inteiros, já que o limite do bloco já é 'até a
    próxima chave' (não mais 'só a 1ª linha')."""
    valor = " ".join(bloco.split())
    return "" if valor == "[Não Informado]" else valor


def handler_texto(bloco):
    return _texto_limpo(bloco)


def _extrai_label_no_bloco(bloco, label_regex):
    """Dentro de um bloco já delimitado, extrai o valor de um sub-rótulo
    (ex.: 'CNPJ/CPF:' dentro do bloco de 'Razão social do produtor'), até o
    fim da linha ou até outro sub-rótulo conhecido."""
    m = re.search(label_regex, bloco)
    if not m:
        return ""
    resto = bloco[m.end():]
    linhas = resto.split("\n")
    primeira = linhas[0].strip()
    if primeira and not _is_label_line(primeira):
        return primeira
    for linha in linhas[1:]:
        s = linha.strip()
        if not s:
            continue
        return "" if _is_label_line(s) else s
    return ""


def handler_empresa(bloco):
    """'Razão social do produtor' / 'Razão social do exportador': o BLOCO
    INTEIRO até a próxima chave (não só a 1ª linha!) pode conter até 3
    informações — razão social, CNPJ/CPF, Inscrição Estadual — cada uma
    opcional. O que sobra depois de remover as linhas de CNPJ/CPF e
    Inscrição Estadual é a razão social. Nunca inventamos as que faltam."""
    cnpj = _extrai_label_no_bloco(bloco, r"CNPJ/CPF:")
    insc = _extrai_label_no_bloco(bloco, r"Inscrição Estadual:")

    sem_labels = re.sub(r"CNPJ/CPF:.*", "", bloco)
    sem_labels = re.sub(r"Inscrição Estadual:.*", "", sem_labels)
    nome = " ".join(l.strip() for l in sem_labels.split("\n") if l.strip())

    return {"Razão social": nome, "CNPJ/CPF": cnpj, "Inscrição Estadual": insc}


def handler_domicilio(bloco):
    """'Domicílio legal e parque industrial' — usado tanto para o produtor
    quanto para o exportador (a 1ª ocorrência do rótulo no documento vira o
    do produtor, a 2ª o do exportador; ver build_keys)."""
    return {
        "Endereço": _extrai_label_no_bloco(bloco, r"Endereço:"),
        "Tel": _extrai_label_no_bloco(bloco, r"Telefone:"),
        "E-mail": _extrai_label_no_bloco(bloco, r"E-mail:"),
    }


def handler_ncm_naladi_denominacao(bloco):
    """'Código NCM - NALADI: ... Denominação comercial do produto a
    exportar'. Extrai os 2 códigos (NCM e NALADI) e junta o texto restante
    (que mistura a descrição tarifária oficial com a denominação comercial
    digitada) num único campo — mesmo formato do exemplo de referência do
    usuário (json djo.txt)."""
    codigos = re.findall(r"\d{4}\.\d{2}\.\d{2}", bloco)
    ncm = codigos[0] if len(codigos) > 0 else ""
    naladi = codigos[1] if len(codigos) > 1 else ""

    sem_header = re.sub(r"Denominação comercial do produto a exportar", " ", bloco)
    for c in codigos[:2]:
        sem_header = sem_header.replace(c, " ", 1)

    return {
        "Código NCM": ncm,
        "NALADI": naladi,
        "Denominação comercial do produto a exportar": " ".join(sem_header.split()),
    }


# --- tabelas de materiais ----------------------------------------------------
def _tokenize_columns(line):
    """Separa uma linha em 'colunas' com base em blocos de 2+ espaços,
    devolvendo os objetos de match (para termos posição inicial/final
    exatas). Frases com espaço único (ex.: 'ITÁLIA', 'SORO DE LEITE') ficam
    num único token."""
    return list(re.finditer(r"\S+(?:[ \t]\S+)*", line))


# Nomes de país (sempre em maiúsculas nas tabelas de materiais dessas DJOs),
# usados APENAS como fallback em _parse_item_row para separar Descrição de
# País origem quando a OCR não deixa o espaçamento duplo de costume entre
# as duas colunas (ex.: "...Outros. CHILE" com um único espaço). Lista não
# exaustiva de todos os países do mundo — cobre Mercosul e parceiros
# comerciais comuns; um país fora da lista simplesmente não aciona o
# fallback (comportamento igual ao anterior a esta correção, nunca pior).
KNOWN_COUNTRIES = {
    "BRASIL", "ARGENTINA", "PARAGUAI", "URUGUAI", "VENEZUELA", "BOLÍVIA",
    "CHILE", "COLÔMBIA", "PERU", "EQUADOR", "CHINA", "ITÁLIA", "ALEMANHA",
    "FRANÇA", "ESPANHA", "PORTUGAL", "JAPÃO", "ÍNDIA", "MÉXICO", "CANADÁ",
    "HOLANDA", "BÉLGICA", "SUÍÇA", "ÁUSTRIA", "RÚSSIA", "TAIWAN", "VIETNÃ",
    "TAILÂNDIA", "INDONÉSIA", "MALÁSIA", "TURQUIA", "ISRAEL", "AUSTRÁLIA",
    "CUBA", "PANAMÁ", "POLÔNIA", "HUNGRIA", "ROMÊNIA", "UCRÂNIA",
    "DINAMARCA", "SUÉCIA", "NORUEGA", "FINLÂNDIA", "IRLANDA", "GRÉCIA",
    "EGITO", "MARROCOS", "NIGÉRIA", "CINGAPURA", "FILIPINAS",
    "ESTADOS UNIDOS", "REINO UNIDO", "COREIA DO SUL", "COREIA DO NORTE",
    "PAÍSES BAIXOS", "ÁFRICA DO SUL", "NOVA ZELÂNDIA", "COSTA RICA",
    "EL SALVADOR", "REPÚBLICA DOMINICANA", "REPÚBLICA TCHECA",
    "ARÁBIA SAUDITA", "EMIRADOS ÁRABES UNIDOS", "HONG KONG",
}
_COUNTRY_WORD_COUNTS = sorted({len(c.split()) for c in KNOWN_COUNTRIES}, reverse=True)


def _split_descricao_pais(merged_text):
    """Fallback usado só quando Descrição e País origem chegam grudados num
    único token (sem o gap de 2+ espaços). Procura, no FINAL de
    `merged_text`, um nome de país conhecido (1 ou 2 palavras) e separa ali;
    caso nenhum seja encontrado, devolve tudo como Descrição e País origem
    vazio — exatamente o comportamento de antes desta correção."""
    words = merged_text.split(" ")
    for word_count in _COUNTRY_WORD_COUNTS:
        if len(words) <= word_count:
            continue
        tail = " ".join(words[-word_count:])
        if tail.upper() in KNOWN_COUNTRIES:
            return " ".join(words[:-word_count]).strip(), tail
    return merged_text.strip(), ""


def _parse_item_row(line, ncm_token_match):
    """Faz o parsing de UMA linha que inicia um item da tabela.

    Descrição / País origem continuam separados por espaçamento (2+
    espaços) sempre que possível; quando a OCR só deixa 1 espaço entre elas
    (token único), tentamos separar um país conhecido do final do texto via
    _split_descricao_pais antes de desistir. Valor (US$) / % S/VALOR FOB /
    Fornecedor-Fabricante são separados por CONTEÚDO: localizamos o ÚLTIMO
    padrão 'número%' da linha (0%, 5%, 12.5%, 25,50%, 100%... —
    inconfundível por causa do '%') — tudo depois dele é
    Fornecedor/Fabricante, mesmo colado sem espaço nenhum ou grudado num
    texto padrão tipo "0% DISPONIBLE À SOLITUD DE LAS AUTORIDADES
    COMPETENTES". Do que sobra antes do '%', o último número (ou '[Não
    Informado]') é o Valor (US$); o que sobra antes disso é Descrição +
    País origem.
    """
    ncm = ncm_token_match.group().strip()
    base = ncm_token_match.end()
    resto_bruto = line[base:]

    pct_matches = list(re.finditer(r"[\d.,]+\s*%", resto_bruto))
    if pct_matches:
        pct_m = pct_matches[-1]
        pct_val = re.sub(r"\s+", "", pct_m.group())
        antes_pct = resto_bruto[:pct_m.start()]
        pos_pct_abs = base + pct_m.start()
        pos_fornecedor_abs = base + pct_m.end()
        fornecedor = resto_bruto[pct_m.end():].strip()
    else:
        pct_val = ""
        antes_pct = resto_bruto
        pos_pct_abs = base + len(resto_bruto)
        pos_fornecedor_abs = pos_pct_abs
        fornecedor = ""

    valor_matches = list(re.finditer(r"\[Não Informado\]|[\d.,]+", antes_pct))
    if valor_matches:
        valor_m = valor_matches[-1]
        valor_val = valor_m.group()
        antes_valor = antes_pct[:valor_m.start()]
        pos_valor_abs = base + valor_m.start()
    else:
        valor_val = ""
        antes_valor = antes_pct
        pos_valor_abs = base + len(antes_pct)

    subtokens = _tokenize_columns(antes_valor)
    if len(subtokens) >= 2:
        descricao = subtokens[0].group().strip()
        pais = " ".join(t.group().strip() for t in subtokens[1:])
        pos_descricao_abs = base + subtokens[0].start()
        pos_pais_abs = base + subtokens[1].start()
    elif len(subtokens) == 1:
        merged = subtokens[0].group().strip()
        descricao, pais = _split_descricao_pais(merged)
        pos_descricao_abs = base + subtokens[0].start()
        if pais:
            # país é o sufixo de `merged`; sua posição absoluta é o início
            # do token único mais o deslocamento até onde o sufixo começa
            pos_pais_abs = pos_descricao_abs + (len(merged) - len(pais))
        else:
            pos_pais_abs = pos_valor_abs
    else:
        descricao, pais = "", ""
        pos_descricao_abs = pos_pais_abs = base

    item = {
        "NCM/SH": ncm,
        "Descrição": descricao,
        "País origem": pais,
        "Valor (US$)": valor_val,
        "% s/Valor FOB": pct_val,
        "Fornecedor/Fabricante": fornecedor,
    }
    ancoras = [
        ("Descrição", pos_descricao_abs),
        ("País origem", pos_pais_abs),
        ("Valor (US$)", pos_valor_abs),
        ("% s/Valor FOB", pos_pct_abs),
        ("Fornecedor/Fabricante", pos_fornecedor_abs),
    ]
    return item, ancoras


def handler_tabela(texto_secao):
    """Parser de uma das 4 seções de materiais (tabela de itens + Somatório)."""
    linhas = texto_secao.split("\n")
    nao_vazias = [l for l in linhas if l.strip()]

    resultado = {"Items": [], "Somatório": ""}
    if not nao_vazias:
        return resultado
    if len(nao_vazias) == 1 and "Não Informado" in nao_vazias[0]:
        return resultado

    m_som = re.search(r"Somat.rio:\s*([\d.,]+%)", texto_secao)
    if m_som:
        resultado["Somatório"] = m_som.group(1)

    header_idx = next((i for i, l in enumerate(linhas) if "NCM/SH" in l), None)
    if header_idx is None:
        return resultado

    header_line = linhas[header_idx]
    ncm_col_start = len(header_line) - len(header_line.lstrip())

    item_atual = None
    ancoras_atuais = None
    itens = []

    for linha in linhas[header_idx + 1:]:
        if not linha.strip():
            continue
        if re.search(r"Somat.rio:\s*[\d.,]+%", linha):
            break
        if "NCM/SH" in linha:
            continue

        indent = len(linha) - len(linha.lstrip())
        ncm_m = re.match(r"^\s*(\d{4}\.\d{2}\.\d{2}|\[Não Informado\])", linha)
        nova_linha = ncm_m is not None and abs(indent - ncm_col_start) <= 3

        if nova_linha:
            if item_atual:
                itens.append(item_atual)
            item_atual, ancoras_atuais = _parse_item_row(linha, ncm_m)
        else:
            if item_atual is None or not ancoras_atuais:
                continue
            for tok in _tokenize_columns(linha):
                texto_tok = tok.group().strip()
                melhor_campo = min(ancoras_atuais, key=lambda a: abs(a[1] - tok.start()))[0]
                if item_atual[melhor_campo]:
                    item_atual[melhor_campo] += " " + texto_tok
                else:
                    item_atual[melhor_campo] = texto_tok

    if item_atual:
        itens.append(item_atual)

    # Blindagem: "% s/Valor FOB" só deve conter um valor percentual (dígitos
    # + separador decimal opcional + "%"). Se alguma linha de continuação
    # vazou texto extra para essa coluna (ex.: uma linha de totais que a OCR
    # grafou de um jeito não previsto pelos regex acima), descartamos tudo
    # depois do valor percentual em vez de propagar o lixo para o JSON.
    for item in itens:
        m_pct = re.match(r"\s*([\d.,]+%)", item["% s/Valor FOB"])
        if m_pct:
            item["% s/Valor FOB"] = m_pct.group(1)

    resultado["Items"] = itens
    return resultado


# ---------------------------------------------------------------------------
# 3) LISTA ORDENADA DE CHAVES — a ordem é a ordem real do documento
# ---------------------------------------------------------------------------
def build_keys():
    return [
        {"name": "numero_djo", "pattern": r"Número da DJO:", "handler": handler_texto},
        {"name": "data_apresentacao", "pattern": r"Data de apresentação:", "handler": handler_texto},
        {"name": "razao_social_produtor", "pattern": r"Razão social do produtor:", "handler": handler_empresa},
        {"name": "domicilio_produtor", "pattern": r"Domicílio legal e parque industrial:", "handler": handler_domicilio},
        {"name": "razao_social_exportador", "pattern": r"Razão social do exportador:", "handler": handler_empresa},
        {"name": "domicilio_exportador", "pattern": r"Domicílio legal e parque industrial:", "handler": handler_domicilio},
        {"name": "ncm_naladi_denominacao", "pattern": r"Código NCM\s*-\s*NALADI:", "handler": handler_ncm_naladi_denominacao, "merge": True},
        {"name": "valor_fob_usd", "pattern": r"Valor FOB \(USD\):", "handler": handler_texto},
        {"name": "unidade_medida", "pattern": r"Unidade de medida:", "handler": handler_texto},
        {"name": "descricao_processo_produtivo", "pattern": r"Descrição do processo produtivo:", "handler": handler_texto},
        {"name": "materiais_utilizados_header", "pattern": r"Materiais utilizados:", "handler": None},
        {"name": "originarios_estado_parte_produtor", "pattern": r"Materiais originários do Estado Parte produtor:", "handler": handler_tabela},
        {"name": "originarios_outros_estados_partes", "pattern": r"Materiais originários de outros Estados Partes:", "handler": handler_tabela},
        {"name": "nao_originarios", "pattern": r"Materiais não originários:", "handler": handler_tabela},
        {"name": "terceiros_paises_ptc", "pattern": r"Materiais de terceiros países que tenham cumprido com a PTC:", "handler": handler_tabela},
        {"name": "porcentagem_total_materias_primas", "pattern": r"Porcentagem Total de Mat[eé]rias Primas, Componentes ou Partes:", "handler": handler_texto},
        {"name": "valor_agregado_processo_industrial", "pattern": r"Valor agregado no processo Industrial \(Deduzidos os tributos restituídos ou a restituir em caso de exportação\):", "handler": handler_texto},
        {"name": "preco_fob", "pattern": r"Preço FOB:", "handler": handler_texto},
        {"name": "observacoes", "pattern": r"Observações:", "handler": handler_texto},
    ]


# ---------------------------------------------------------------------------
# 4) ORQUESTRAÇÃO + MONTAGEM DO JSON FINAL
# ---------------------------------------------------------------------------
def parsear_djo(extracted_text):
    """Retorna (json_final, debug_blocks). debug_blocks é usado na célula de
    depuração logo abaixo para inspecionar o que cada chave capturou."""
    codigo_aprovacao, texto_limpo = preprocess_ocr_text(extracted_text)
    campos, blocos_debug = sequential_extract(texto_limpo, build_keys())

    vazio_tabela = {"Items": [], "Somatório": ""}

    json_final = {
        "Código da aprovação DJO": codigo_aprovacao,
        "Número da DJO": campos.get("numero_djo", ""),
        "Data de apresentação": campos.get("data_apresentacao", ""),
        "Razão social do produtor": campos.get("razao_social_produtor", {}),
        "Domicílio legal e parque industrial do produtor": campos.get("domicilio_produtor", {}),
        "Razão social do exportador": campos.get("razao_social_exportador", {}),
        "Domicílio legal e parque industrial do exportador": campos.get("domicilio_exportador", {}),
        "Código NCM": campos.get("Código NCM", ""),
        "NALADI": campos.get("NALADI", ""),
        "Denominação comercial do produto a exportar": campos.get("Denominação comercial do produto a exportar", ""),
        "Valor FOB (USD)": campos.get("valor_fob_usd", ""),
        "Unidade de medida": campos.get("unidade_medida", ""),
        "Descrição do processo produtivo": campos.get("descricao_processo_produtivo", ""),
        "Materiales": {
            "originarios_estado_parte_produtor": campos.get("originarios_estado_parte_produtor", vazio_tabela),
            "originarios_outros_estados_partes": campos.get("originarios_outros_estados_partes", vazio_tabela),
            "nao_originarios": campos.get("nao_originarios", vazio_tabela),
            "terceiros_paises_ptc": campos.get("terceiros_paises_ptc", vazio_tabela),
            "Porcentagem Total de Matérias Primas, Componentes ou Partes": campos.get("porcentagem_total_materias_primas", ""),
            "Valor agregado no processo Industrial (Deduzidos os tributos restituídos ou a restituir em caso de exportação)": campos.get("valor_agregado_processo_industrial", ""),
            "Preço FOB": campos.get("preco_fob", ""),
        },
        "Observações": campos.get("observacoes", ""),
    }
    return json_final, blocos_debug


# --- Ejemplo de Ejecución ---
json_final, debug_blocks = parsear_djo(extracted_text)
print(json.dumps(json_final, indent=4, ensure_ascii=False))

{
    "Código da aprovação DJO": "",
    "Número da DJO": "5645787",
    "Data de apresentação": "",
    "Razão social do produtor": {
        "Razão social": "INCASA S/A",
        "CNPJ/CPF": "84.689.090/0002-40",
        "Inscrição Estadual": ""
    },
    "Domicílio legal e parque industrial do produtor": {
        "Endereço": "ESTRADA DONA FRANCISCA, 11700, 89239-270 PIRABEIRABA -, JOINVILLE -SC",
        "Tel": "(47) 3205-7000",
        "E-mail": "camily.stolf@incasa.ind.br"
    },
    "Razão social do exportador": {
        "Razão social": "Incasa S/A",
        "CNPJ/CPF": "84.689.090/0001-60",
        "Inscrição Estadual": "250781360"
    },
    "Domicílio legal e parque industrial do exportador": {
        "Endereço": "Rua Dona Francisca, 11.700 - Bairro PIRABEIRABA - CEP 89239-270 - JOINVILLE - SC - BRASIL",
        "Tel": "4732057000",
        "E-mail": "camila.kricheldorf@incasa.ind.br"
    },
    "Código NCM": "2827.60.12",
    "NALADI": "2827.60.20",
    "Denominação comer

In [78]:
# =============================================================================
# DEPURAÇÃO: inspecionar o bloco bruto extraído para cada chave, na ordem em
# que aparecem no documento, ANTES de virarem o JSON final. Use isto para
# validar a extração campo a campo quando algo no JSON final parecer errado.
# =============================================================================
LARGURA_PREVIEW = 300  # aumente se precisar ver blocos de tabela inteiros

for b in debug_blocks:
    status = "OK" if b["found"] else "RÓTULO NÃO ENCONTRADO"
    print(f"### {b['key']}  [{status}]")
    if b["found"]:
        print(f"  rótulo casado : {b['matched_label']!r}")
        bloco = b["raw_block"]
        preview = bloco if len(bloco) <= LARGURA_PREVIEW else bloco[:LARGURA_PREVIEW] + " …(truncado)"
        print(f"  bloco bruto   : {preview!r}")
    print("-" * 100)

### numero_djo  [OK]
  rótulo casado : 'Número da DJO:'
  bloco bruto   : ' \n5645787 \n'
----------------------------------------------------------------------------------------------------
### data_apresentacao  [OK]
  rótulo casado : 'Data de apresentação:'
  bloco bruto   : ' \n\n'
----------------------------------------------------------------------------------------------------
### razao_social_produtor  [OK]
  rótulo casado : 'Razão social do produtor:'
  bloco bruto   : ' \nINCASA S/A \nCNPJ/CPF: 84.689.090/0002-40 \n'
----------------------------------------------------------------------------------------------------
### domicilio_produtor  [OK]
  rótulo casado : 'Domicílio legal e parque industrial:'
  bloco bruto   : ' \nEndereço: ESTRADA DONA FRANCISCA, 11700, 89239-270 PIRABEIRABA -, JOINVILLE -SC \n\nTelefone: (47) 3205-7000 \nE-mail: camily.stolf@incasa.ind.br \n'
----------------------------------------------------------------------------------------------------
### ra

In [79]:
json_final['Materiales'].keys()

dict_keys(['originarios_estado_parte_produtor', 'originarios_outros_estados_partes', 'nao_originarios', 'terceiros_paises_ptc', 'Porcentagem Total de Matérias Primas, Componentes ou Partes', 'Valor agregado no processo Industrial (Deduzidos os tributos restituídos ou a restituir em caso de exportação)', 'Preço FOB'])

In [80]:
import pandas as pd
pd.DataFrame(json_final['Materiales']['originarios_estado_parte_produtor']['Items'])

,NCM/SH,Descrição,País origem,Valor (US$),% s/Valor FOB,Fornecedor/Fabricante
0,2815.20.00,Hidróxido de sódio (soda cáustica); hidróxido ...,BRASIL,"4,85","16,17%",Indefinido


In [81]:
pd.DataFrame(json_final['Materiales']['originarios_outros_estados_partes']['Items'])

,NCM/SH,Descrição,País origem,Valor (US$),% s/Valor FOB,Fornecedor/Fabricante
0,2801.20.90,"Flúor, cloro, bromo e iodo. lodo. Outros.",CHILE,"10,15","33,83%",Indefinido


In [82]:
json_final['Materiales']['originarios_estado_parte_produtor']['Somatório']

'16,17%'

In [83]:
json_final['Materiales']['originarios_outros_estados_partes']['Somatório']

'33,83%'

In [85]:
extracted_text

'\n\n                                                        DECLARAÇÃO JURAMENTADA DE ORIGEM \n\n                                                                                                                                                         Código da aprovação DJO: \n\nNúmero da DJO: \n5645787 \nData de apresentação: \n\nRazão social do produtor: \nINCASA S/A \nCNPJ/CPF: 84.689.090/0002-40 \nDomicílio legal e parque industrial: \nEndereço: ESTRADA DONA FRANCISCA, 11700, 89239-270 PIRABEIRABA -, JOINVILLE -SC \n\nTelefone: (47) 3205-7000 \nE-mail: camily.stolf@incasa.ind.br \nRazão social do exportador: \nIncasa S/A \nCNPJ/CPF: 84.689.090/0001-60 \nInscrição Estadual: 250781360 \nDomicílio legal e parque industrial: \nEndereço: Rua Dona Francisca, 11.700 - Bairro PIRABEIRABA - CEP 89239-270 - JOINVILLE - SC - BRASIL \nTelefone: 4732057000 \nE-mail: camila.kricheldorf@incasa.ind.br \n\nCódigo NCM - NALADI:                                                                     Deno